# OCCAM Python Interface (pyoccam) Demonstration Notebook

This notebook demonstrates the complete workflow for using OCCAM through Python:
1. Loading data (with automatic test data detection)
2. Running search algorithms
3. Selecting best models by different criteria
4. Generating detailed fit reports
5. Analyzing results

**Note:** Run cells in order for best results.

In [1]:
# Import required libraries
import pyoccam
import sys
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Check pyoccam2 version
print(f"pyoccam2 version: {pyoccam.__version__}")
print(f"Python version: {sys.version}")
print(f"Notebook run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

AttributeError: module 'pyoccam' has no attribute '__version__'

In [2]:
# CONFIGURATION - Modify these settings as needed

# Data Configuration
DATA_FILE = "SY_sample_pts_to_occam3_shuffle_split42_hdr.txt"

# Search Configuration
SEARCH_TYPE = "loopless-up"    # Options: loopless-up, full-up, disjoint-up, chain-up
SEARCH_LEVELS = 7
SEARCH_WIDTH = 3
ALPHA_THRESHOLD = 0.05

# Model Selection
BEST_MODEL_CRITERION = "bic"    # Options: bic, aic, information, info_alpha

# Output Configuration
OUTPUT_DIR = "notebook_output"
OUTPUT_FORMAT = "space"         # Options: space, csv, tab, html

# Fit Report Configuration
GENERATE_FIT_REPORT = True
FIT_TARGET_STATE = "0"
SKIP_RESIDUALS = False
SKIP_IVI_TABLES = False

# Debug
DEBUG_MODE = False

print("Configuration loaded successfully")
print(f"Data file: {DATA_FILE}")
print(f"Search type: {SEARCH_TYPE}")
print(f"Model selection: {BEST_MODEL_CRITERION}")

Configuration loaded successfully
Data file: SY_sample_pts_to_occam3_shuffle_split42_hdr.txt
Search type: loopless-up
Model selection: bic


In [3]:
# Helper functions

def create_output_directory():
    """Create output directory if it doesn't exist"""
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Created output directory: {OUTPUT_DIR}")
    else:
        print(f"Output directory exists: {OUTPUT_DIR}")

def get_separator_constant(format_name):
    """Convert format name to pyoccam2 separator constant"""
    separators = {
        "tab": pyoccam2.TABSEP,
        "csv": pyoccam2.COMMASEP,
        "space": pyoccam2.SPACESEP,
        "html": pyoccam2.HTMLFORMAT
    }
    return separators.get(format_name.lower(), pyoccam2.SPACESEP)

def save_output(content, filename):
    """Save output to file"""
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, 'w') as f:
        f.write(content)
    print(f"Saved to: {filepath}")
    return filepath

# Create output directory
create_output_directory()

Created output directory: notebook_output


## Step 1: Initialize OCCAM and Load Data

In [4]:
# Initialize OCCAM manager
manager = pyoccam2.VBMManager()

# Enable debug mode if configured
if DEBUG_MODE:
    manager.set_debug_mode(True)
    print("Debug mode: ENABLED")

# Load data file
print(f"Loading: {DATA_FILE}")
success = manager.init_from_command_line(["occam", DATA_FILE])

if not success:
    raise Exception(f"Failed to load data file '{DATA_FILE}'")

print("✓ Data loaded successfully")

Loading: SY_sample_pts_to_occam3_shuffle_split42_hdr.txt
✓ Data loaded successfully


In [5]:
# Display basic statistics
stats = manager.get_basic_statistics()
print("Data Statistics:")
print(stats)

# Check for test data
has_test = manager.has_test_data()
if has_test:
    print("\n✓ Test data detected - will include test performance metrics")
else:
    print("\nℹ No test data found - using training data only")

# Get variable information
variables = manager.get_variable_list()
print(f"\nVariables ({len(variables)}):")
if len(variables) <= 10:
    for i, var in enumerate(variables):
        print(f"  {i+1:2d}. {var}")
else:
    # Show first and last few
    for i in range(3):
        print(f"  {i+1:2d}. {variables[i]}")
    print("  ...")
    for i in range(len(variables)-2, len(variables)):
        print(f"  {i+1:2d}. {variables[i]}")

Data Statistics:
Sample size: 1077
Variables: 21
H(data): 10.7334
Test data: Present


✓ Test data detected - will include test performance metrics

Variables (21):
   1. ASpect_reclass
   2. CLay_reclass
   3. CurVature_reclass
  ...
  20. DrainageCl
  21. LS


## Step 2: Configure OCCAM Settings

In [6]:
# Configure OCCAM settings

# Set reference model
manager.set_ref_model("bottom")
print("Reference model: bottom (independence)")

# Set output format
separator = get_separator_constant(OUTPUT_FORMAT)
manager.set_report_separator(separator)
print(f"Output format: {OUTPUT_FORMAT}")

# Configure fit report options
if GENERATE_FIT_REPORT:
    manager.set_fit_classifier_target(FIT_TARGET_STATE)
    manager.set_skip_trained_model_table(SKIP_RESIDUALS)
    manager.set_skip_ivi_tables(SKIP_IVI_TABLES)
    print(f"Fit report target state: {FIT_TARGET_STATE}")

print("\n✓ Configuration complete")

Reference model: bottom (independence)
Output format: space
Fit report target state: 0

✓ Configuration complete


## Step 3: Run Search Algorithm

In [7]:
# Run search algorithm
print(f"Running {SEARCH_TYPE} search...")
print(f"  Levels: {SEARCH_LEVELS}")
print(f"  Width: {SEARCH_WIDTH}")
print(f"  Test data: {'included' if has_test else 'not available'}")

# Generate search report
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=has_test
)

print("\n✓ Search complete")
print(f"Models evaluated: {manager.get_search_model_count()}")

Running loopless-up search...
  Levels: 7
  Width: 3
  Test data: included

✓ Search complete
Models evaluated: 22


In [8]:
# Display first part of search report (first 50 lines)
lines = search_report.split('\n')
print(f"Search Report Preview (first 50 lines of {len(lines)} total):")
print("-" * 80)
for line in lines[:50]:
    print(line)
print("-" * 80)
print(f"... ({len(lines)-50} more lines)")

Search Report Preview (first 50 lines of 47 total):
--------------------------------------------------------------------------------
Searching levels:
1 : 20 new models, 3 kept; 4 total kept
2 : 54 new models, 3 kept; 7 total kept
3 : 52 new models, 3 kept; 10 total kept
4 : 48 new models, 3 kept; 13 total kept
5 : 45 new models, 3 kept; 16 total kept
6 : 43 new models, 3 kept; 19 total kept
7 : 40 new models, 3 kept; 22 total kept

  ID   MODEL                        Level              H            ddf            dLR          Alpha        %dH(DV)           daic           dbic      Inc.Alpha       %C(Data)         %cover       %C(Test)          %miss
  22   IV:ClElFdGlLcSlTcZ               7        10.3139            287       626.4231         0.0000        46.8290        52.4231     -1377.3921         1.0000        78.8301        47.9167        67.5373         7.0896
  21   IV:ElFdGlLcSlTcDcZ               7        10.3361            287       593.2203         0.0000        44.3469   

In [9]:
# Save search report
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
search_filename = f"{SEARCH_TYPE}_search_{timestamp}.txt"
search_path = save_output(search_report, search_filename)

# Also save as CSV
if OUTPUT_FORMAT != "csv":
    manager.set_report_separator(pyoccam2.COMMASEP)
    csv_report = manager.generate_search_report(
        search_type=SEARCH_TYPE,
        levels=SEARCH_LEVELS,
        width=SEARCH_WIDTH,
        include_test_data=has_test
    )
    csv_filename = f"{SEARCH_TYPE}_search_{timestamp}.csv"
    csv_path = save_output(csv_report, csv_filename)
    # Reset separator
    manager.set_report_separator(separator)
    print(f"CSV version saved: {csv_filename}")

Saved to: notebook_output\loopless-up_search_20250831_142817.txt
Saved to: notebook_output\loopless-up_search_20250831_142817.csv
CSV version saved: loopless-up_search_20250831_142817.csv


## Step 4: Analyze Search Results and Select Best Model

In [10]:
# Get best models by different criteria
best_models = {
    "bic": manager.get_best_model_by_bic(),
    "aic": manager.get_best_model_by_aic(),
    "information": manager.get_best_model_by_information(),
    "info_alpha": manager.get_best_model_by_info_alpha()
}

print("Best models found:")
for criterion, model_name in best_models.items():
    if model_name:
        print(f"  {criterion.upper():12s}: {model_name}")
    else:
        print(f"  {criterion.upper():12s}: (none found)")

Best models found:
  BIC         : IV:LcTcZ
  AIC         : IV:HbLcTcZ
  INFORMATION : IV:ClElFdGlLcSlTcZ
  INFO_ALPHA  : IV:HbLcTcZ


In [11]:
# Select model based on configured criterion
selected_model = best_models.get(BEST_MODEL_CRITERION)

if not selected_model:
    print(f"WARNING: No model found using criterion '{BEST_MODEL_CRITERION}'")
    selected_model = best_models.get("information")
    if selected_model:
        print(f"Falling back to best by information: {selected_model}")

if not selected_model:
    raise Exception("No models found in search results")

print(f"\n✓ Selected model (by {BEST_MODEL_CRITERION}): {selected_model}")


✓ Selected model (by bic): IV:LcTcZ


In [12]:
# Get detailed statistics for selected model
model_stats = manager.get_model_statistics(selected_model)

print(f"Selected model statistics for: {selected_model}")
print("-" * 50)
print(f"  Entropy (H):        {model_stats.h:.4f}")
print(f"  Information:        {model_stats.information:.4f} ({model_stats.information*100:.2f}%)")
print(f"  Degrees of freedom: {model_stats.df:.0f}")
print(f"  Likelihood ratio:   {model_stats.lr:.4f}")
print(f"  Alpha (p-value):    {model_stats.alpha:.6f}")
print(f"  AIC:                {model_stats.aic:.4f}")
print(f"  BIC:                {model_stats.bic:.4f}")
print(f"  Delta AIC:          {model_stats.daic:.4f}")
print(f"  Delta BIC:          {model_stats.dbic:.4f}")

if has_test:
    print(f"\nTest Data Performance:")
    print(f"  % Correct (train):  {model_stats.pct_correct_data:.2f}%")
    print(f"  % Correct (test):   {model_stats.pct_correct_test:.2f}%")
    print(f"  % Coverage:         {model_stats.pct_coverage:.2f}%")
    print(f"  % Missed (test):    {model_stats.pct_missed_test:.2f}%")
else:
    print(f"  % Correct (data):   {model_stats.pct_correct_data:.2f}%")

Selected model statistics for: IV:LcTcZ
--------------------------------------------------
  Entropy (H):        10.4672
  Information:        0.2972 (29.72%)
  Degrees of freedom: 8
  Likelihood ratio:   397.5253
  Alpha (p-value):    0.000000
  AIC:                381.5253
  BIC:                341.6698
  Delta AIC:          381.5253
  Delta BIC:          341.6698

Test Data Performance:
  % Correct (train):  71.96%
  % Correct (test):   63.81%
  % Coverage:         100.00%
  % Missed (test):    0.00%


## Step 5: Generate Detailed Fit Report (Optional)

In [14]:
if GENERATE_FIT_REPORT:
    print(f"Generating fit report for: {selected_model}")
    print(f"Target state: {FIT_TARGET_STATE}")

    # Generate fit report
    fit_report = manager.generate_fit_report(
        model_name=selected_model,
        target_state=FIT_TARGET_STATE
    )

    # Save fit report
    fit_filename = f"fit_{selected_model.replace(':', '_')}_{timestamp}.txt"
    fit_path = save_output(fit_report, fit_filename)

    print("\n✓ Fit report generated")
    print("Includes:")
    print("  • Model statistics")
    print("  • Contingency table")
    if not SKIP_RESIDUALS:
        print("  • Residual analysis")
    if not SKIP_IVI_TABLES:
        print("  • IVI tables")
    print("  • Confusion matrix")
    if has_test:
        print("  • Test data performance")
else:
    print("Fit report generation skipped (GENERATE_FIT_REPORT = False)")

Generating fit report for: IV:LcTcZ
Target state: 0
Saved to: notebook_output\fit_IV_LcTcZ_20250831_142817.txt

✓ Fit report generated
Includes:
  • Model statistics
  • Contingency table
  • Residual analysis
  • IVI tables
  • Confusion matrix
  • Test data performance


In [15]:
if GENERATE_FIT_REPORT:
    # Display first part of fit report
    fit_lines = fit_report.split('\n')
    print(f"\nFit Report Preview (first 30 lines of {len(fit_lines)} total):")
    print("-" * 80)
    for line in fit_lines[:30]:
        print(line)
    print("-" * 80)
    print(f"... ({len(fit_lines)-30} more lines)")


Fit Report Preview (first 30 lines of 57 total):
--------------------------------------------------------------------------------
Sample size: 1077
Variables: 21
Test data: Present

    Model,IV:LcTcZ (Directed System)
    IV Component:,ASpect_reclass; CLay_reclass; CurVature_reclass; ELevation_reclass; Fault_Dns_reclass; HaBitatMap_UTM; Geol_Lith; nlcd_2021_Land_Cover; Geol_rock_Type; Rd_strm_Dns_reclass; SLope_reclass; TWi_reclass; GeomDesc; TaxOrder; TaxSuborder; taxGrtGroup; TaxSubgrp; taxPartSize; TaxClname; DrainageCl,AsClCvElFdHbGlLcGtRdSlTwGdToTsGgTs1PsTcDc
    Model Component: ,nlcd_2021_Land_Cover; TaxClname; LS,LcTcZ
    Degrees of Freedom (DF):,3.0611e+08
    Loops:,NO
    Entropy(H):,10.4672
    Information captured (%):,29.7175
    Transmission (T):,0.629692

-------------------------------------------------------------------------

    REFERENCE = TOP
    ,Value,Prob. (Alpha)
    Log-Likelihood (LR),940.155,1
    Pearson X2,619.537,
    Delta DF (dDF),3.0611e+08,

-----

## Summary and Next Steps

In [16]:
# Analysis summary
print("="*60)
print("ANALYSIS COMPLETE")
print("="*60)

print("\nSummary:")
print(f"  • Data: {DATA_FILE}")
print(f"  • Sample size: {manager.get_sample_size()}")
print(f"  • Variables: {len(variables)}")
print(f"  • Search: {SEARCH_TYPE}")
print(f"  • Models evaluated: {manager.get_search_model_count()}")
print(f"  • Best model: {selected_model}")
print(f"  • Selection: {BEST_MODEL_CRITERION}")

print(f"\nOutput files in: {OUTPUT_DIR}/")

print("\nNext steps:")
print("  1. Review search report for model space")
print("  2. Examine fit report for interpretation")
print("  3. Try alternative search types")
print("  4. Compare selection criteria")
if has_test:
    print("  5. Validate on test data")

ANALYSIS COMPLETE

Summary:
  • Data: SY_sample_pts_to_occam3_shuffle_split42_hdr.txt
  • Sample size: 1077
  • Variables: 21
  • Search: loopless-up
  • Models evaluated: 22
  • Best model: IV:LcTcZ
  • Selection: bic

Output files in: notebook_output/

Next steps:
  1. Review search report for model space
  2. Examine fit report for interpretation
  3. Try alternative search types
  4. Compare selection criteria
  5. Validate on test data
